<a href="https://colab.research.google.com/" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctica: Detección de Cáncer de Piel con Deep Learning Multimodal

**Basado en**: Saeed et al. (2025). *Multimodal deep learning ensemble framework for skin cancer detection*. Scientific Reports.

**Dataset**: HAM10000 — Skin Cancer MNIST (equivalente al ISIC 2018 del artículo)

**Idea principal del artículo**: combinar imágenes dermatoscópicas con metadatos clínicos del paciente (edad, sexo, localización) mejora significativamente la clasificación frente a usar solo imágenes.

---

### Estructura
1. Instalación y configuración del entorno
2. Descarga del dataset (Kaggle API)
3. Carga y exploración de datos
4. Preprocesamiento (imágenes + metadatos)
5. Construcción del modelo multimodal (MobileNetV2 + metadatos)
6. Entrenamiento
7. Evaluación (accuracy, F1, matriz de confusión)
8. Demo de predicción

>  **Importante**: Activar GPU antes de empezar → *Entorno de ejecución → Cambiar tipo → T4 GPU*

## 1. Instalación y configuración del entorno

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

In [ ]:
!pip install -q kaggle

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
import keras
from keras import layers, Model
from keras.applications import MobileNetV2
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

# Semilla para reproducibilidad
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow: {tf.__version__}")
print(f"Keras:      {keras.__version__}")
print(f"GPU:        {tf.config.list_physical_devices('GPU')}")

## 2. Descarga del dataset (Kaggle API)

HAM10000 contiene **10.015 imágenes dermatoscópicas** de 7 tipos de lesiones cutáneas junto con metadatos clínicos del paciente. Es el mismo dataset ISIC 2018 utilizado en el artículo.

In [ ]:
# Subimos el fichero kaggle.json
# Descárgalo en: kaggle.com → Account → API → Create New Token
from google.colab import files
print("Sube tu fichero kaggle.json:")
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Descarga (~3.5 GB, tarda unos minutos)
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 --unzip -p /content/ham10000

print("\n Descarga completada")
!ls /content/ham10000

## 3. Carga y exploración de datos

In [ ]:
# Cargamos el CSV de metadatos
meta = pd.read_csv('/content/ham10000/HAM10000_metadata.csv')
print(f"Registros totales: {meta.shape[0]}")
print(f"Columnas: {list(meta.columns)}")
meta.head()

In [ ]:
# Construimos un diccionario image_id → ruta del fichero
# Las imágenes están repartidas en dos carpetas (part_1 y part_2)
img_paths = glob.glob('/content/ham10000/**/*.jpg', recursive=True)
img_dict  = {Path(p).stem: p for p in img_paths}

meta['image_path'] = meta['image_id'].map(img_dict)
meta = meta.dropna(subset=['image_path']).reset_index(drop=True)

print(f"Imágenes encontradas: {len(img_dict)}")
print(f"Registros con imagen válida: {len(meta)}")

In [ ]:
# Nombres completos de cada clase (igual que el artículo)
NOMBRES_CLASES = {
    'mel':   'Melanoma',
    'nv':    'Nevo Melanocítico',
    'bcc':   'Carcinoma Basocelular',
    'akiec': 'Queratosis Actínica',
    'bkl':   'Queratosis Benigna',
    'df':    'Dermatofibroma',
    'vasc':  'Lesión Vascular'
}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Distribución de clases (desbalance evidente)
counts = meta['dx'].value_counts()
axes[0].bar(counts.index, counts.values, color='steelblue', edgecolor='black')
axes[0].set_title('Distribución de clases', fontsize=12)
axes[0].set_xlabel('Tipo de lesión')
axes[0].set_ylabel('N.º de imágenes')
axes[0].tick_params(axis='x', rotation=30)

# Distribución de edad
axes[1].hist(meta['age'].dropna(), bins=20, color='coral', edgecolor='black')
axes[1].set_title('Distribución de edad', fontsize=12)
axes[1].set_xlabel('Edad')

# Proporción de sexo
sexo = meta['sex'].value_counts()
axes[2].pie(sexo, labels=sexo.index, autopct='%1.1f%%',
            colors=['skyblue', 'lightpink', 'lightgray'])
axes[2].set_title('Proporción de sexo', fontsize=12)

plt.suptitle('Exploración del dataset HAM10000', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n⚠️  Desbalance claro: NV tiene ~6700 imágenes, DF y VASC solo ~100.")
print("    → Usaremos class_weight para compensarlo durante el entrenamiento.")

In [ ]:
# Mostramos una imagen de ejemplo por cada clase
clases = meta['dx'].unique()
fig, axes = plt.subplots(1, len(clases), figsize=(16, 3))

for ax, cls in zip(axes, clases):
    fila = meta[meta['dx'] == cls].iloc[0]
    img  = plt.imread(fila['image_path'])
    ax.imshow(img)
    ax.set_title(cls.upper(), fontsize=9, fontweight='bold')
    ax.axis('off')

plt.suptitle('Muestra de cada tipo de lesión cutánea', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Preprocesamiento

Procesamos los dos tipos de datos por separado:

**Imágenes**: redimensionado a 224×224 px y normalización al rango [0, 1].

**Metadatos clínicos**: igual que en el artículo, usamos edad (normalizada), sexo y localización anatómica.

In [ ]:
# ----- Metadatos -----
# Imputamos valores ausentes
meta['age']          = meta['age'].fillna(meta['age'].mean())
meta['sex']          = meta['sex'].fillna('unknown')
meta['localization'] = meta['localization'].fillna('unknown')

# Codificamos etiquetas de clase (mel, nv, bcc…) → 0, 1, 2…
le = LabelEncoder()
meta['label'] = le.fit_transform(meta['dx'])
CLASS_NAMES  = list(le.classes_)           # ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
NUM_CLASSES  = len(CLASS_NAMES)
print(f"Clases ({NUM_CLASSES}): {CLASS_NAMES}")

# Codificamos sexo: male→0, female→1, unknown→2
sex_map = {'male': 0, 'female': 1, 'unknown': 2}
meta['sex_enc'] = meta['sex'].map(sex_map).fillna(2)

# Codificamos localización anatómica con LabelEncoder
le_loc = LabelEncoder()
meta['loc_enc'] = le_loc.fit_transform(meta['localization'])

# Normalizamos la edad al rango [0, 1]
meta['age_norm'] = meta['age'] / 100.0

# Vector de metadatos final: 3 características (igual que el notebook de referencia)
META_COLS = ['age_norm', 'sex_enc', 'loc_enc']
print(f"Metadatos usados: {META_COLS}")
meta[META_COLS].describe()

In [ ]:
# División 80 / 10 / 10 (igual que el artículo)
train_df, temp_df = train_test_split(
    meta, test_size=0.2, stratify=meta['label'], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label'], random_state=SEED
)

print(f"Entrenamiento: {len(train_df):5d} imágenes")
print(f"Validación:    {len(val_df):5d} imágenes")
print(f"Test:          {len(test_df):5d} imágenes")

In [ ]:
# ----- Class weight -----
# El desbalance hace que el modelo tienda a predecir siempre NV.
# class_weight penaliza más los errores en clases minoritarias.
# Aplicamos raíz cuadrada para suavizar pesos muy extremos.

cw = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['label']),
    y=train_df['label']
)
# sqrt suaviza: DF pasaría de ~9.8 a ~3.1 (mismo efecto, más estable)
class_weight_dict = {i: np.sqrt(w) for i, w in enumerate(cw)}

print("Pesos por clase (suavizados con √):")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name:8s}: {class_weight_dict[i]:.3f}")

In [ ]:
# ----- Pipelines tf.data -----
# Construimos datasets eficientes que cargan imagen y metadatos juntos.

IMG_SIZE   = 224
BATCH_SIZE = 32

def cargar_imagen(path):
    """Lee, decodifica y normaliza una imagen a 224×224."""
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0   # normalización [0, 1]
    return img

def aumentar(img):
    """Augmentación aleatoria solo para entrenamiento."""
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, max_delta=0.15)
    img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img

def crear_dataset(df, augmentacion=False, shuffle=True):
    """Crea un tf.data.Dataset con entradas (img, meta) y etiqueta."""
    paths  = df['image_path'].values
    metas  = df[META_COLS].values.astype(np.float32)
    labels = df['label'].values.astype(np.int32)

    def procesar(path, meta, label):
        img = cargar_imagen(path)
        if augmentacion:
            img = aumentar(img)
        # Devolvemos un diccionario de entradas y la etiqueta
        return {'img_input': img, 'meta_input': meta}, label

    ds = tf.data.Dataset.from_tensor_slices((paths, metas, labels))
    ds = ds.map(procesar, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=1000, seed=SEED)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

ds_train = crear_dataset(train_df, augmentacion=True,  shuffle=True)
ds_val   = crear_dataset(val_df,   augmentacion=False, shuffle=False)
ds_test  = crear_dataset(test_df,  augmentacion=False, shuffle=False)

print(" Datasets creados")
print(f"   Tamaño de metadatos por muestra: {len(META_COLS)} características")

## 5. Construcción del modelo multimodal

El modelo tiene **dos ramas paralelas** que se fusionan mediante concatenación, siguiendo la arquitectura del artículo:

```
Rama imagen:    MobileNetV2 (ImageNet) → GlobalAvgPool → Dense(128) → BN → Dropout(0.5)
                                                                              ↓
Rama metadatos: Dense(32) → Dropout(0.3) → Dense(16)              → Concatenate
                                                                              ↓
                                                              Dense(128) → Dropout(0.4)
                                                                              ↓
                                                              Dense(7, softmax)
```

Usamos **MobileNetV2** en lugar de EfficientNetB0 porque es más ligero (~2.3M parámetros vs ~5.3M) y converge más rápido en pocas épocas, ideal para la práctica.

In [ ]:
def construir_modelo_multimodal(num_clases=NUM_CLASSES, img_size=IMG_SIZE,
                                 meta_dim=len(META_COLS)):
    """
    Modelo multimodal: imagen dermatoscópica + metadatos clínicos.
    Arquitectura basada en Saeed et al. (2025), Fig. 8.

    Args:
        num_clases: número de tipos de lesión (7 en HAM10000)
        img_size:   tamaño de la imagen de entrada (224)
        meta_dim:   número de características de metadatos (3)

    Returns:
        model: modelo Keras compilado listo para entrenar
    """

    # ── Rama de imagen ──────────────────────────────────────────────────────
    # MobileNetV2 pre-entrenada en ImageNet, sin la cabeza clasificadora
    base = MobileNetV2(
        include_top=False,
        weights='imagenet',
        input_shape=(img_size, img_size, 3)
    )
    base.trainable = False  # Congelamos los pesos de ImageNet

    img_input = keras.Input(shape=(img_size, img_size, 3), name='img_input')
    x = base(img_input, training=False)        # Extracción de características
    x = layers.GlobalAveragePooling2D()(x)     # Reduce (7,7,1280) → (1280,)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)          # Estabiliza el entrenamiento
    x = layers.Dropout(0.5)(x)                 # Regularización

    # ── Rama de metadatos ───────────────────────────────────────────────────
    meta_input = keras.Input(shape=(meta_dim,), name='meta_input')
    m = layers.Dense(32, activation='relu')(meta_input)
    m = layers.Dropout(0.3)(m)
    m = layers.Dense(16, activation='relu')(m)

    # ── Fusión: concatenamos imagen + metadatos ──────────────────────────────
    fusionado = layers.Concatenate()([x, m])   # (128 + 16 = 144 neuronas)
    fusionado = layers.Dense(128, activation='relu')(fusionado)
    fusionado = layers.Dropout(0.4)(fusionado)

    # ── Capa de salida ───────────────────────────────────────────────────────
    salida = layers.Dense(num_clases, activation='softmax', name='output')(fusionado)

    model = Model(
        inputs=[img_input, meta_input],
        outputs=salida,
        name='modelo_multimodal_cancer_piel'
    )
    return model


modelo = construir_modelo_multimodal()

modelo.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

modelo.summary()

## 6. Entrenamiento

Usamos dos callbacks para controlar el entrenamiento automáticamente:
- **EarlyStopping**: para el entrenamiento si `val_loss` no mejora en 5 épocas
- **ReduceLROnPlateau**: reduce el learning rate si `val_loss` se estanca

In [ ]:
EPOCHS = 15

callbacks = [
    # Monitoramos val_loss: más estable que val_accuracy con clases desbalanceadas
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,         # lr_nuevo = lr_actual × 0.3
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

print("Iniciando entrenamiento...")
print(f"  Épocas máx.: {EPOCHS}")
print(f"  Batch size:  {BATCH_SIZE}")
print(f"  Learning rate inicial: 1e-3")
print()

history = modelo.fit(
    ds_train,
    validation_data=ds_val,
    epochs=EPOCHS,
    class_weight=class_weight_dict,  # Compensa el desbalance de clases
    callbacks=callbacks
)

print("\n Entrenamiento completado")

In [ ]:
# Curvas de entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

epocas = range(1, len(history.history['loss']) + 1)

# Loss
axes[0].plot(epocas, history.history['loss'],     label='Entrenamiento', color='steelblue')
axes[0].plot(epocas, history.history['val_loss'], label='Validación',    color='coral', linestyle='--')
axes[0].set_title('Función de pérdida (Loss)', fontsize=12)
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epocas, history.history['accuracy'],     label='Entrenamiento', color='steelblue')
axes[1].plot(epocas, history.history['val_accuracy'], label='Validación',    color='coral', linestyle='--')
axes[1].set_title('Exactitud (Accuracy)', fontsize=12)
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Curvas de entrenamiento — Modelo multimodal', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Evaluación

Evaluamos el modelo en el conjunto de test (datos que nunca ha visto durante el entrenamiento).

In [ ]:
# Evaluación general
loss_test, acc_test = modelo.evaluate(ds_test, verbose=0)
print(f"Test Loss:     {loss_test:.4f}")
print(f"Test Accuracy: {acc_test:.4f}  ({acc_test*100:.2f}%)")

In [ ]:
# Obtenemos predicciones sobre el test set
y_probs = modelo.predict(ds_test, verbose=0)
y_pred  = np.argmax(y_probs, axis=1)
y_true  = test_df['label'].values

print("Informe de clasificación por clase:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=[c.upper() for c in CLASS_NAMES],
    yticklabels=[c.upper() for c in CLASS_NAMES]
)
plt.title('Matriz de Confusión — Modelo Multimodal (imagen + metadatos)', fontsize=12)
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.tight_layout()
plt.show()

In [ ]:
# Guardamos el modelo entrenado
modelo.save('cancer_piel_multimodal.keras')
print(" Modelo guardado como 'cancer_piel_multimodal.keras'")

## 8. Demo de predicción

Probamos el modelo con una imagen del conjunto de test y los datos clínicos del paciente.

In [ ]:
# Seleccionamos una muestra aleatoria del test set para demostrar la predicción
idx        = np.random.randint(0, len(test_df))
fila       = test_df.iloc[idx]

# Preparamos la imagen
img_raw    = tf.io.read_file(fila['image_path'])
img_raw    = tf.image.decode_jpeg(img_raw, channels=3)
img_224    = tf.image.resize(img_raw, [IMG_SIZE, IMG_SIZE])
img_norm   = tf.cast(img_224, tf.float32) / 255.0
img_batch  = tf.expand_dims(img_norm, axis=0)  # (1, 224, 224, 3)

# Preparamos los metadatos
meta_vals  = np.array([[fila['age_norm'], fila['sex_enc'], fila['loc_enc']]],
                       dtype=np.float32)

# Predicción
probs      = modelo.predict(
    {'img_input': img_batch, 'meta_input': meta_vals}, verbose=0
)
pred_idx   = np.argmax(probs)
confianza  = np.max(probs) * 100
pred_clase = CLASS_NAMES[pred_idx]
real_clase = CLASS_NAMES[fila['label']]

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Imagen
axes[0].imshow(img_raw.numpy())
axes[0].axis('off')
color_titulo = 'green' if pred_clase == real_clase else 'red'
axes[0].set_title(
    f"Real: {real_clase.upper()}\nPredicho: {pred_clase.upper()} ({confianza:.1f}%)",
    color=color_titulo, fontsize=12, fontweight='bold'
)

# Probabilidades por clase
colores = ['coral' if c == pred_clase else 'steelblue' for c in CLASS_NAMES]
axes[1].barh(CLASS_NAMES, probs[0], color=colores)
axes[1].set_xlim(0, 1)
axes[1].set_xlabel('Probabilidad')
axes[1].set_title('Probabilidad por clase', fontsize=12)
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle(
    f"Paciente: {fila['sex']} | Edad: {fila['age']:.0f} años | Localización: {fila['localization']}",
    fontsize=11
)
plt.tight_layout()
plt.show()

print(f"\n→ Clase real:     {real_clase}")
print(f"→ Clase predicha: {pred_clase}")
print(f"→ Confianza:      {confianza:.2f}%")
print(f"→ ¿Correcto?:     {'✅ Sí' if pred_clase == real_clase else ' No'}")

## Conclusiones

Este cuaderno implementa los elementos clave del artículo de Saeed et al. (2025):

| Elemento | Implementación |
|---|---|
| Transfer Learning | MobileNetV2 pre-entrenada en ImageNet |
| Metadatos clínicos | Edad, sexo y localización anatómica |
| Fusión multimodal | Concatenación de características visuales y clínicas |
| Desbalance de clases | `class_weight` con suavizado por raíz cuadrada |
| Augmentación | Flips, brillo, contraste aleatorios |
| Regularización | Dropout(0.5) y BatchNormalization |

**Observaciones**:
- Las clases minoritarias (DF, VASC, AKIEC) obtienen F1 más bajos incluso con `class_weight` porque tienen muy pocas muestras
- La fusión de metadatos mejora la clasificación en lesiones visualmente similares (MEL vs BKL)
- Con más épocas y fine-tuning de las últimas capas de MobileNetV2, los resultados mejorarían hacia los del artículo